In [ ]:
# ============================================================
# BOOTSTRAP CELL -- run this first.
# Installs anything missing, detects CUDA / MPS / CPU, wires the
# local sam2/ and build.py into sys.path. Works on Windows,
# Linux, macOS, and Google Colab without any extra step.
# ============================================================
import importlib, shutil, subprocess, sys, os, platform
from pathlib import Path

# 1) locate the bundle root (walk up until we find build.py)
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'build.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'build.py').exists():
    raise RuntimeError('Could not find build.py by walking up from the notebook folder. '
                       'Open the notebook from inside the bundle folder.')
os.environ['SAM2_PROJECT_ROOT'] = str(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT =', PROJECT_ROOT)

# 2) helper -- tries system pip first, falls back to --user
def _pip(pkgs, extra=()):
    cmd = [sys.executable, '-m', 'pip', 'install', *pkgs, *extra]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', *pkgs, *extra])

# 3) make sure torch + torchvision are there with the right wheel
def _detect_cuda():
    exe = shutil.which('nvidia-smi')
    if not exe: return None
    try:
        out = subprocess.check_output([exe], stderr=subprocess.STDOUT, timeout=8).decode(errors='ignore')
        for line in out.splitlines():
            if 'CUDA Version' in line:
                return line.split('CUDA Version:')[1].strip().split()[0]
    except Exception:
        pass
    return None

try:
    import torch  # noqa: F401
except ImportError:
    sys_name = platform.system()
    in_colab = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ
    if in_colab:
        pass  # Colab already ships torch with CUDA
    elif _detect_cuda() and sys_name in ('Windows', 'Linux'):
        print('NVIDIA GPU detected -> installing CUDA torch wheel ...')
        _pip(['torch', 'torchvision'], extra=['--index-url', 'https://download.pytorch.org/whl/cu121'])
    elif sys_name == 'Darwin':
        print('macOS detected -> installing default torch (MPS/CPU) ...')
        _pip(['torch', 'torchvision'])
    else:
        print('No GPU -> installing CPU-only torch ...')
        _pip(['torch', 'torchvision'], extra=['--index-url', 'https://download.pytorch.org/whl/cpu'])

# 4) install everything else that is missing
_needed = {
    'hydra':            'hydra-core',
    'omegaconf':        'omegaconf',
    'iopath':           'iopath',
    'huggingface_hub':  'huggingface_hub',
    'cv2':              'opencv-python',
    'matplotlib':       'matplotlib',
    'ipympl':           'ipympl',
    'PIL':              'pillow',
    'tqdm':             'tqdm',
}
_missing = []
for mod, pkg in _needed.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        _missing.append(pkg)
if _missing:
    print('Installing missing packages:', _missing)
    _pip(_missing)
    print('If any import still fails below, restart the kernel and rerun this cell.')

# 5) report the torch backend
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
    os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')
else:
    device = torch.device('cpu')
print('torch', torch.__version__, '| device:', device)


In [ ]:
# Make sure hydra & iopath are present
import sys, subprocess
for pkg in ('hydra-core', 'iopath'):
    try:
        __import__(pkg.split('-')[0].replace('-', '_'))
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])

# Make the local sam2 package importable whatever the OS
from pathlib import Path
import os
PROJECT_ROOT = Path(os.environ.get('SAM2_PROJECT_ROOT', Path.cwd())).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import sam2
from sam2.modeling.backbones import hieradet
print('sam2 loaded from:', sam2.__file__)
